# Risk-Sensitive Reinforcement Learning for Trading
**Comparison: PPO vs CVaR-PPO vs Sortino-PPO**

Author: Student Project  
Date: 2026  
Environment: Kaggle GPU

**Project Overview:**  
Implement and compare three reinforcement learning methods for algorithmic trading:
1. PPO (Risk-Neutral Baseline)
2. CVaR-PPO (Tail Risk Constraint)
3. Sortino-PPO (Downside Deviation Penalty)

Against baseline: Buy and Hold strategy

**Key improvements over v1:**
- Multi-asset data: SPY, QQQ, GLD
- Crisis-period evaluation (COVID 2020, Bear 2022)
- Fixed CVaR loss computation (normalized returns)
- Added Sortino-PPO as third method
- Extended metrics: Calmar, Sortino ratio, VaR, CVaR of returns

## 1. Setup

In [ ]:
import subprocess, sys

for pkg in ['yfinance', 'ta', 'gymnasium', 'torch', 'pandas', 'numpy', 'matplotlib', 'plotly']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('All packages ready')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf
import ta
import gymnasium as gym
from gymnasium import spaces
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Normal
import json
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
CONFIG = {
    # Data
    'SYMBOLS': ['SPY', 'QQQ', 'GLD'],     # S&P500, Nasdaq, Gold
    'PRIMARY_SYMBOL': 'SPY',
    'START_DATE': '2018-01-01',
    'END_DATE':   '2024-12-31',
    'INTERVAL': '1d',

    # Data split by market regime (not fixed ratio)
    # Train: 2018-2021 (includes COVID crash 2020)
    # Val:   2022      (bear market -19%)
    # Test:  2023-2024 (recovery + bull run)
    'TRAIN_END': '2021-12-31',
    'VAL_END':   '2022-12-31',

    # Crisis periods for stress testing
    'CRISIS_PERIODS': {
        'COVID_crash': ('2020-02-01', '2020-05-31'),
        'Bear_2022':   ('2022-01-01', '2022-12-31'),
        'Bull_2023':   ('2023-01-01', '2024-12-31'),
    },

    # Environment
    'INITIAL_BALANCE': 10000,
    'TRANSACTION_COST': 0.001,
    'SLIPPAGE': 0.0005,

    # PPO (shared architecture)
    'HIDDEN_DIM': 256,
    'GAMMA': 0.99,
    'PPO_LR': 3e-4,
    'PPO_EPSILON': 0.2,
    'PPO_EPOCHS': 10,

    # CVaR-PPO
    'CVAR_LR': 2e-4,
    'CVAR_ALPHA': 0.15,         # Worst 15% of returns
    'CVAR_LAMBDA': 0.12,        # CVaR penalty weight
    'CVAR_LAMBDA_DECAY': 0.997,
    'CVAR_LAMBDA_MIN': 0.05,
    'CVAR_L2': 1e-5,

    # Sortino-PPO
    'SORTINO_LR': 2e-4,
    'SORTINO_LAMBDA': 0.5,      # Downside penalty weight

    # Training
    'NUM_EPISODES': 150,
    'UPDATE_INTERVAL': 512,

    'DEVICE': 'cuda' if torch.cuda.is_available() else 'cpu'
}

print('Configuration:')
print(f'  Assets: {CONFIG["SYMBOLS"]}')
print(f'  Train: {CONFIG["START_DATE"]} to {CONFIG["TRAIN_END"]}')
print(f'  Val:   {CONFIG["TRAIN_END"]} to {CONFIG["VAL_END"]}')
print(f'  Test:  {CONFIG["VAL_END"]} to {CONFIG["END_DATE"]}')
print(f'  Device: {CONFIG["DEVICE"]}')

## 2. Data Loading

In [ ]:
def download_data(symbol, start_date, end_date):
    """Download and validate OHLCV data from Yahoo Finance."""
    ticker = yf.Ticker(symbol)
    df = ticker.history(start=start_date, end=end_date, interval='1d')

    if df.empty:
        raise ValueError(f'No data for {symbol}')

    df.reset_index(inplace=True)
    df.columns = df.columns.str.lower()
    df = df[['date', 'open', 'high', 'low', 'close', 'volume']]

    # Basic cleaning
    df = df[~df['date'].duplicated(keep='first')]
    df = df[(df[['open','high','low','close']] > 0).all(axis=1)]
    df.dropna(inplace=True)
    df.sort_values('date', inplace=True)
    df.reset_index(drop=True, inplace=True)

    return df


all_data = {}
for sym in CONFIG['SYMBOLS']:
    df = download_data(sym, CONFIG['START_DATE'], CONFIG['END_DATE'])
    all_data[sym] = df
    print(f'{sym}: {len(df)} rows | {df["date"].min().date()} to {df["date"].max().date()}')
    print(f'       Close: min=${df["close"].min():.2f}, max=${df["close"].max():.2f}, mean=${df["close"].mean():.2f}')

raw_data = all_data[CONFIG['PRIMARY_SYMBOL']]
print(f'\nPrimary asset for training: {CONFIG["PRIMARY_SYMBOL"]} ({len(raw_data)} rows)')

## 3. Feature Engineering

In [ ]:
def add_features(df):
    """Add technical indicators. Returns cleaned DataFrame."""
    d = df.copy()

    # Trend
    d['sma_10']  = ta.trend.sma_indicator(d['close'], window=10)
    d['sma_20']  = ta.trend.sma_indicator(d['close'], window=20)
    d['sma_50']  = ta.trend.sma_indicator(d['close'], window=50)

    # Momentum
    d['rsi'] = ta.momentum.rsi(d['close'], window=14)
    macd = ta.trend.MACD(d['close'])
    d['macd']        = macd.macd()
    d['macd_signal'] = macd.macd_signal()
    d['macd_diff']   = macd.macd_diff()

    # Volatility
    bb = ta.volatility.BollingerBands(d['close'])
    d['bb_high'] = bb.bollinger_hband()
    d['bb_low']  = bb.bollinger_lband()
    d['bb_mid']  = bb.bollinger_mavg()
    d['atr']     = ta.volatility.average_true_range(d['high'], d['low'], d['close'])

    # Returns
    d['returns']     = d['close'].pct_change()
    d['log_returns'] = np.log(d['close'] / d['close'].shift(1))

    # Volume
    d['volume_sma'] = ta.trend.sma_indicator(d['volume'], window=20)

    # Clean
    d.replace([np.inf, -np.inf], np.nan, inplace=True)
    d.dropna(inplace=True)
    d.reset_index(drop=True, inplace=True)

    return d


featured_data = {}
for sym in CONFIG['SYMBOLS']:
    featured_data[sym] = add_features(all_data[sym])
    print(f'{sym}: {len(featured_data[sym])} rows, {len(featured_data[sym].columns)} features')

data = featured_data[CONFIG['PRIMARY_SYMBOL']]
print(f'\nFeatures: {[c for c in data.columns if c not in ["date","open","high","low","close","volume"]]}')

## 4. Data Split by Market Regime

In [ ]:
def split_by_date(df, train_end, val_end):
    """Split data chronologically by date string."""
    # Normalize date column to timezone-naive for comparison
    dates = df['date'].dt.tz_localize(None) if df['date'].dt.tz is not None else df['date']

    t_end = pd.Timestamp(train_end)
    v_end = pd.Timestamp(val_end)

    train = df[dates <= t_end].copy().reset_index(drop=True)
    val   = df[(dates > t_end) & (dates <= v_end)].copy().reset_index(drop=True)
    test  = df[dates > v_end].copy().reset_index(drop=True)

    return train, val, test


train_data, val_data, test_data = split_by_date(
    data, CONFIG['TRAIN_END'], CONFIG['VAL_END']
)

print(f'Train: {len(train_data):4d} rows | {train_data["date"].iloc[0].date()} to {train_data["date"].iloc[-1].date()}')
print(f'Val:   {len(val_data):4d} rows | {val_data["date"].iloc[0].date()} to {val_data["date"].iloc[-1].date()}')
print(f'Test:  {len(test_data):4d} rows | {test_data["date"].iloc[0].date()} to {test_data["date"].iloc[-1].date()}')
print()
print('Market regime mapping:')
print('  Train (2018-2021): includes COVID crash Feb-Mar 2020 (-34%)')
print('  Val   (2022):      full bear market (-19%)')
print('  Test  (2023-2024): recovery + bull run')

## 5. Trading Environment

In [ ]:
class TradingEnv(gym.Env):
    """
    Custom trading environment.

    State:  normalized technical features + portfolio info (balance, shares, value)
    Action: continuous [-1, 1]  (-1=sell all, 0=hold, +1=buy all)
    Reward: step portfolio return (overridden by subclasses for risk-sensitive variants)
    """
    def __init__(self, df, initial_balance=10000, tc=0.001, slippage=0.0005):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.initial_balance = initial_balance
        self.tc = tc
        self.slippage = slippage

        self.feat_cols = [c for c in df.columns
                          if c not in ['date','open','high','low']]
        self.feat_mean = df[self.feat_cols].mean().values
        self.feat_std  = df[self.feat_cols].std().values + 1e-8

        n = len(self.feat_cols) + 3
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(n,), dtype=np.float32)
        self.action_space      = spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32)

        self.reset()

    def reset(self, seed=None, options=None):
        self.step_idx  = 0
        self.balance   = self.initial_balance
        self.shares    = 0.0
        self.port_val  = self.initial_balance
        self.history   = []
        return self._obs(), {}

    def _obs(self):
        raw = self.df.loc[self.step_idx, self.feat_cols].values.astype(np.float32)
        norm = (raw - self.feat_mean) / self.feat_std
        price = float(self.df.loc[self.step_idx, 'close'])
        port  = np.array([
            self.balance / self.initial_balance,
            self.shares * price / self.initial_balance,
            self.port_val / self.initial_balance
        ], dtype=np.float32)
        return np.concatenate([norm, port])

    def step(self, action):
        price = float(self.df.loc[self.step_idx, 'close'])
        a = float(action[0])
        old_val = self.balance + self.shares * price

        if a > 0.01:
            invest = self.balance * a
            exec_price = price * (1 + self.slippage)
            bought = invest * (1 - self.tc) / exec_price
            self.shares  += bought
            self.balance -= invest
        elif a < -0.01:
            sell = self.shares * abs(a)
            exec_price = price * (1 - self.slippage)
            proceeds = sell * exec_price * (1 - self.tc)
            self.shares  -= sell
            self.balance += proceeds

        self.step_idx += 1
        done = self.step_idx >= len(self.df) - 1

        new_price = float(self.df.loc[self.step_idx, 'close'])
        new_val   = self.balance + self.shares * new_price
        self.port_val = new_val

        reward = self._reward(old_val, new_val)

        self.history.append({
            'step': self.step_idx, 'action': a,
            'price': price, 'portfolio_value': new_val, 'reward': reward
        })

        return self._obs(), reward, done, False, {}

    def _reward(self, old_val, new_val):
        """Base reward: simple portfolio return. Override in subclasses."""
        return (new_val - old_val) / (old_val + 1e-10)


# Quick sanity check
env = TradingEnv(train_data, CONFIG['INITIAL_BALANCE'])
obs, _ = env.reset()
print(f'Observation shape: {obs.shape}')
print(f'Action space: {env.action_space}')
print(f'Obs mean: {obs.mean():.4f}, std: {obs.std():.4f}')
print('TradingEnv OK')

## 6. Actor-Critic Network (shared)

In [ ]:
class ActorCritic(nn.Module):
    """Shared backbone Actor-Critic network for all PPO variants."""

    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU()
        )
        self.actor_mean    = nn.Linear(hidden_dim, action_dim)
        self.actor_log_std = nn.Parameter(torch.zeros(action_dim))
        self.critic        = nn.Linear(hidden_dim, 1)

    def act(self, state):
        h    = self.shared(state)
        mean = self.actor_mean(h)
        std  = torch.exp(self.actor_log_std)
        dist = Normal(mean, std)
        raw  = dist.sample()
        lp   = dist.log_prob(raw).sum(-1)
        action = torch.tanh(raw)
        return action, lp

    def evaluate(self, state, action):
        h    = self.shared(state)
        mean = self.actor_mean(h)
        std  = torch.exp(self.actor_log_std)
        dist = Normal(mean, std)
        raw  = torch.atanh(torch.clamp(action, -0.999, 0.999))
        lp   = dist.log_prob(raw).sum(-1)
        lp  -= torch.log(1 - action**2 + 1e-6).sum(-1)
        val  = self.critic(h)
        ent  = dist.entropy().sum(-1)
        return lp, val, ent


state_dim  = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
net = ActorCritic(state_dim, action_dim, CONFIG['HIDDEN_DIM'])
total_params = sum(p.numel() for p in net.parameters())
print(f'ActorCritic: state_dim={state_dim}, action_dim={action_dim}')
print(f'Total parameters: {total_params:,}')

## 7. Base PPO Agent

In [ ]:
class PPOAgent:
    """Proximal Policy Optimization (Risk-Neutral)."""

    def __init__(self, env, lr, config):
        self.env    = env
        self.config = config
        self.device = config['DEVICE']

        s_dim = env.observation_space.shape[0]
        a_dim = env.action_space.shape[0]
        self.policy    = ActorCritic(s_dim, a_dim, config['HIDDEN_DIM']).to(self.device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)

        self.gamma   = config['GAMMA']
        self.epsilon = config['PPO_EPSILON']
        self.epochs  = config['PPO_EPOCHS']

        self._clear_memory()
        self.history = {'rewards': [], 'port_values': [],
                        'policy_losses': [], 'value_losses': []}

    def _clear_memory(self):
        self.mem = {'states':[], 'actions':[], 'rewards':[],
                    'log_probs':[], 'values':[], 'dones':[]}

    def select_action(self, state):
        s = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            action, lp = self.policy.act(s)
            val = self.policy.critic(self.policy.shared(s)).item()
        return action.cpu().numpy()[0], lp.cpu().item(), val

    def store(self, state, action, reward, lp, val, done):
        self.mem['states'].append(state)
        self.mem['actions'].append(action)
        self.mem['rewards'].append(reward)
        self.mem['log_probs'].append(lp)
        self.mem['values'].append(val)
        self.mem['dones'].append(done)

    def _compute_returns(self):
        R, returns = 0, []
        for i in reversed(range(len(self.mem['rewards']))):
            if self.mem['dones'][i]:
                R = 0
            R = self.mem['rewards'][i] + self.gamma * R
            returns.insert(0, R)
        rets = torch.tensor(returns, dtype=torch.float32).to(self.device)
        vals = torch.tensor(self.mem['values'], dtype=torch.float32).to(self.device)
        adv  = rets - vals
        adv  = (adv - adv.mean()) / (adv.std() + 1e-8)
        return rets, adv

    def update(self):
        rets, adv = self._compute_returns()
        states  = torch.FloatTensor(np.array(self.mem['states'])).to(self.device)
        actions = torch.FloatTensor(np.array(self.mem['actions'])).to(self.device)
        old_lp  = torch.FloatTensor(self.mem['log_probs']).to(self.device)

        p_losses, v_losses = [], []
        for _ in range(self.epochs):
            lp, vals, ent = self.policy.evaluate(states, actions)
            vals = vals.squeeze()
            ratio = torch.exp(lp - old_lp)
            s1 = ratio * adv
            s2 = torch.clamp(ratio, 1-self.epsilon, 1+self.epsilon) * adv
            p_loss = -torch.min(s1, s2).mean()
            v_loss = F.mse_loss(vals, rets)
            loss   = p_loss + 0.5*v_loss - 0.01*ent.mean()
            loss  += self._extra_loss(rets)

            self.optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
            self.optimizer.step()

            p_losses.append(p_loss.item())
            v_losses.append(v_loss.item())

        self._clear_memory()
        return np.mean(p_losses), np.mean(v_losses)

    def _extra_loss(self, rets):
        """Hook for risk-sensitive subclasses to add penalty."""
        return torch.tensor(0.0)

    def train(self, num_episodes, label='PPO'):
        print(f'Training {label} for {num_episodes} episodes...')
        total_steps = 0
        for ep in range(num_episodes):
            state, _ = self.env.reset()
            ep_reward = 0
            done = False
            while not done:
                action, lp, val = self.select_action(state)
                next_state, reward, done, _, _ = self.env.step(action)
                self.store(state, action, reward, lp, val, done)
                state = next_state
                ep_reward += reward
                total_steps += 1
                if total_steps % CONFIG['UPDATE_INTERVAL'] == 0:
                    pl, vl = self.update()
                    self.history['policy_losses'].append(pl)
                    self.history['value_losses'].append(vl)

            self.history['rewards'].append(ep_reward)
            self.history['port_values'].append(self.env.port_val)

            if (ep+1) % 30 == 0:
                avg_r = np.mean(self.history['rewards'][-30:])
                avg_p = np.mean(self.history['port_values'][-30:])
                print(f'  [{label}] Ep {ep+1}/{num_episodes}: '
                      f'Avg Reward={avg_r:+.4f}, Avg Portfolio=${avg_p:,.2f}')

        print(f'{label} training done. Final portfolio: ${self.env.port_val:,.2f}')
        return self.history

    def evaluate(self, env):
        state, _ = env.reset()
        done = False
        while not done:
            action, _, _ = self.select_action(state)
            state, _, done, _, _ = env.step(action)
        return env.history, env.port_val


print('PPOAgent defined')

## 8. CVaR-PPO Agent

In [ ]:
class CVaRPPOAgent(PPOAgent):
    """
    CVaR-PPO: adds Conditional Value-at-Risk penalty to PPO.

    Key fix vs v1: returns are normalized before CVaR computation
    to prevent CVaR loss from growing unbounded.
    """

    def __init__(self, env, config):
        super().__init__(env, config['CVAR_LR'], config)
        # Rebuild optimizer with L2 regularization
        self.optimizer = optim.Adam(
            self.policy.parameters(),
            lr=config['CVAR_LR'],
            weight_decay=config['CVAR_L2']
        )
        self.alpha      = config['CVAR_ALPHA']
        self.lam        = config['CVAR_LAMBDA']
        self.lam_decay  = config['CVAR_LAMBDA_DECAY']
        self.lam_min    = config['CVAR_LAMBDA_MIN']

        # Learnable threshold for CVaR
        self.eta = nn.Parameter(torch.tensor(0.0, device=self.device))
        self.eta_opt = optim.Adam([self.eta], lr=config['CVAR_LR'])

        self.history['cvar_values'] = []
        self.history['lambda_values'] = []

    def _cvar_loss(self, returns):
        """
        Compute CVaR loss on normalized returns.
        Normalization prevents loss from growing unbounded.
        """
        r_norm = (returns - returns.mean()) / (returns.std() + 1e-8)
        sorted_r, _ = torch.sort(r_norm)
        n_worst = max(1, int(len(sorted_r) * self.alpha))
        cvar = sorted_r[:n_worst].mean()
        # Penalize when CVaR is negative (tail losses)
        loss = torch.relu(-cvar)
        return loss, cvar.item()

    def _extra_loss(self, rets):
        cvar_loss, cvar_val = self._cvar_loss(rets.detach())
        self.history['cvar_values'].append(cvar_val)
        self.history['lambda_values'].append(self.lam)
        return self.lam * cvar_loss

    def update(self):
        pl, vl = super().update()
        # Decay lambda to gradually reduce risk penalty over training
        self.lam = max(self.lam_min, self.lam * self.lam_decay)
        return pl, vl

    def train(self, num_episodes):
        print(f'Alpha={self.alpha} (worst {self.alpha*100:.0f}%), '
              f'Lambda={self.config["CVAR_LAMBDA"]}→{self.lam_min} '
              f'(decay={self.lam_decay})')
        return super().train(num_episodes, label='CVaR-PPO')


print('CVaRPPOAgent defined')

## 9. Sortino-PPO Agent

In [ ]:
class SortinoPPOAgent(PPOAgent):
    """
    Sortino-PPO: penalizes downside deviation in the reward signal.

    Unlike CVaR-PPO (which constrains tail risk in the loss),
    Sortino-PPO shapes the reward directly:
      reward = portfolio_return - lambda * downside_deviation^2

    This is simpler to implement and more stable to train.
    """

    def __init__(self, env, config):
        super().__init__(env, config['SORTINO_LR'], config)
        self.sortino_lam = config['SORTINO_LAMBDA']

    def train(self, num_episodes):
        print(f'Downside penalty lambda={self.sortino_lam}')
        return super().train(num_episodes, label='Sortino-PPO')


class SortinoTradingEnv(TradingEnv):
    """TradingEnv with Sortino-adjusted reward."""

    def __init__(self, df, sortino_lambda, **kwargs):
        super().__init__(df, **kwargs)
        self.sortino_lam = sortino_lambda
        self._returns_buf = []

    def reset(self, seed=None, options=None):
        self._returns_buf = []
        return super().reset(seed=seed, options=options)

    def _reward(self, old_val, new_val):
        r = (new_val - old_val) / (old_val + 1e-10)
        self._returns_buf.append(r)
        # Downside deviation: only penalize negative returns
        neg = [x for x in self._returns_buf if x < 0]
        downside_std = np.std(neg) if len(neg) > 1 else 0.0
        return r - self.sortino_lam * (downside_std ** 2)


print('SortinoPPOAgent and SortinoTradingEnv defined')

## 10. Training — Method 1: PPO

In [ ]:
train_env_ppo = TradingEnv(
    train_data,
    initial_balance=CONFIG['INITIAL_BALANCE'],
    tc=CONFIG['TRANSACTION_COST'],
    slippage=CONFIG['SLIPPAGE']
)

ppo_agent = PPOAgent(train_env_ppo, CONFIG['PPO_LR'], CONFIG)
ppo_hist  = ppo_agent.train(CONFIG['NUM_EPISODES'])

torch.save(ppo_agent.policy.state_dict(), 'ppo_model.pth')
print('PPO model saved to ppo_model.pth')

## 11. Training — Method 2: CVaR-PPO

In [ ]:
train_env_cvar = TradingEnv(
    train_data,
    initial_balance=CONFIG['INITIAL_BALANCE'],
    tc=CONFIG['TRANSACTION_COST'],
    slippage=CONFIG['SLIPPAGE']
)

cvar_agent = CVaRPPOAgent(train_env_cvar, CONFIG)
cvar_hist  = cvar_agent.train(CONFIG['NUM_EPISODES'])

torch.save(cvar_agent.policy.state_dict(), 'cvar_ppo_model.pth')
print('CVaR-PPO model saved to cvar_ppo_model.pth')

## 12. Training — Method 3: Sortino-PPO

In [ ]:
train_env_sortino = SortinoTradingEnv(
    train_data,
    sortino_lambda=CONFIG['SORTINO_LAMBDA'],
    initial_balance=CONFIG['INITIAL_BALANCE'],
    tc=CONFIG['TRANSACTION_COST'],
    slippage=CONFIG['SLIPPAGE']
)

sortino_agent = SortinoPPOAgent(train_env_sortino, CONFIG)
sortino_hist  = sortino_agent.train(CONFIG['NUM_EPISODES'])

torch.save(sortino_agent.policy.state_dict(), 'sortino_ppo_model.pth')
print('Sortino-PPO model saved to sortino_ppo_model.pth')

## 13. Training Progress Visualization

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Episode Rewards', 'Portfolio Value during Training'))

colors = {'PPO': 'blue', 'CVaR-PPO': 'red', 'Sortino-PPO': 'green'}
for name, hist in [('PPO', ppo_hist), ('CVaR-PPO', cvar_hist), ('Sortino-PPO', sortino_hist)]:
    fig.add_trace(go.Scatter(y=hist['rewards'],     name=name, line=dict(color=colors[name])), row=1, col=1)
    fig.add_trace(go.Scatter(y=hist['port_values'], name=name, line=dict(color=colors[name]), showlegend=False), row=1, col=2)

fig.add_hline(y=CONFIG['INITIAL_BALANCE'], line_dash='dash', line_color='gray', row=1, col=2)
fig.update_layout(title='Training Progress', template='plotly_white', width=1100, height=420)
fig.update_xaxes(title_text='Episode')
fig.update_yaxes(title_text='Reward', row=1, col=1)
fig.update_yaxes(title_text='Portfolio Value ($)', row=1, col=2)
fig.show()

for name, hist in [('PPO', ppo_hist), ('CVaR-PPO', cvar_hist), ('Sortino-PPO', sortino_hist)]:
    final_r = hist['rewards'][-1]
    final_p = hist['port_values'][-1]
    ret_pct = (final_p - CONFIG['INITIAL_BALANCE']) / CONFIG['INITIAL_BALANCE'] * 100
    print(f'{name:15s}: Final Portfolio=${final_p:,.2f} ({ret_pct:+.2f}%)')

## 14. Performance Metrics

In [ ]:
def calculate_metrics(history, initial_balance=10000):
    """
    Comprehensive risk-performance metrics.
    Includes: Return, Sharpe, Sortino, Calmar, MaxDD, VaR, CVaR, WinRate, Volatility.
    """
    pv      = np.array([h['portfolio_value'] for h in history])
    returns = np.diff(pv) / (pv[:-1] + 1e-10)

    total_return = (pv[-1] - initial_balance) / initial_balance
    annual_return = (pv[-1] / initial_balance) ** (252 / len(returns)) - 1

    # Volatility
    vol = np.std(returns) * np.sqrt(252)

    # Sharpe (annualized, risk-free=0)
    sharpe = np.mean(returns) / (np.std(returns) + 1e-10) * np.sqrt(252)

    # Sortino (penalize downside only)
    neg = returns[returns < 0]
    down_std = np.std(neg) * np.sqrt(252) if len(neg) > 1 else 1e-10
    sortino = annual_return / down_std

    # Max Drawdown
    peak   = np.maximum.accumulate(pv)
    dd     = (pv - peak) / (peak + 1e-10)
    max_dd = dd.min()

    # Calmar = annual_return / |max_drawdown|
    calmar = annual_return / abs(max_dd) if max_dd != 0 else 0.0

    # VaR and CVaR at 95% confidence
    var_95  = float(np.percentile(returns, 5))
    tail    = returns[returns <= var_95]
    cvar_95 = float(tail.mean()) if len(tail) > 0 else var_95

    win_rate = float((returns > 0).sum() / len(returns))

    return {
        'final_value':   float(pv[-1]),
        'total_return':  float(total_return),
        'annual_return': float(annual_return),
        'sharpe':        float(sharpe),
        'sortino':       float(sortino),
        'calmar':        float(calmar),
        'max_drawdown':  float(max_dd),
        'var_95':        var_95,
        'cvar_95':       cvar_95,
        'volatility':    float(vol),
        'win_rate':      win_rate,
        'num_trades':    len(history),
    }


print('calculate_metrics() defined')
print('Metrics: Return, Annual Return, Sharpe, Sortino, Calmar, MaxDD, VaR95, CVaR95, Volatility, WinRate')

## 15. Buy and Hold Baseline

In [ ]:
def buy_and_hold(df, initial_balance=10000):
    """Buy at open of first day, hold until last day."""
    price0 = float(df['close'].iloc[0])
    shares = initial_balance / price0
    history = []
    for i, row in df.iterrows():
        history.append({
            'step': i,
            'action': 1.0 if i == 0 else 0.0,
            'price': row['close'],
            'portfolio_value': shares * row['close'],
            'reward': 0.0
        })
    return history, shares * float(df['close'].iloc[-1])


bh_test_hist, bh_test_val = buy_and_hold(test_data, CONFIG['INITIAL_BALANCE'])
bh_val_hist,  bh_val_val  = buy_and_hold(val_data,  CONFIG['INITIAL_BALANCE'])
bh_train_hist,bh_train_val= buy_and_hold(train_data, CONFIG['INITIAL_BALANCE'])

bh_m = calculate_metrics(bh_test_hist, CONFIG['INITIAL_BALANCE'])
print(f'Buy & Hold — Train: ${bh_train_val:,.2f} | Val: ${bh_val_val:,.2f} | Test: ${bh_test_val:,.2f}')
print(f'Buy & Hold Test — Return: {bh_m["total_return"]*100:+.2f}%, Sharpe: {bh_m["sharpe"]:.3f}')

## 16. Standard Evaluation on Test Set

In [ ]:
def make_test_env(df, env_class=TradingEnv, **kwargs):
    return env_class(
        df,
        initial_balance=CONFIG['INITIAL_BALANCE'],
        tc=CONFIG['TRANSACTION_COST'],
        slippage=CONFIG['SLIPPAGE'],
        **kwargs
    )


ppo_test_env     = make_test_env(test_data)
cvar_test_env    = make_test_env(test_data)
sortino_test_env = make_test_env(test_data, env_class=SortinoTradingEnv,
                                 sortino_lambda=CONFIG['SORTINO_LAMBDA'])

ppo_hist_test,     _ = ppo_agent.evaluate(ppo_test_env)
cvar_hist_test,    _ = cvar_agent.evaluate(cvar_test_env)
sortino_hist_test, _ = sortino_agent.evaluate(sortino_test_env)

metrics = {
    'PPO':         calculate_metrics(ppo_hist_test,     CONFIG['INITIAL_BALANCE']),
    'CVaR-PPO':    calculate_metrics(cvar_hist_test,    CONFIG['INITIAL_BALANCE']),
    'Sortino-PPO': calculate_metrics(sortino_hist_test, CONFIG['INITIAL_BALANCE']),
    'Buy & Hold':  calculate_metrics(bh_test_hist,      CONFIG['INITIAL_BALANCE']),
}

print('TEST SET RESULTS')
keys = ['final_value','total_return','sharpe','sortino','calmar','max_drawdown','cvar_95','volatility','win_rate']
df_results = pd.DataFrame(metrics).T[keys]
df_results.columns = ['Final Value','Return','Sharpe','Sortino','Calmar','Max DD','CVaR-95','Volatility','Win Rate']
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print(df_results.to_string())

df_results.to_csv('test_results.csv')
print('\nSaved to test_results.csv')

## 17. Crisis Period Stress Test

In [ ]:
def eval_on_period(agent, df, start, end, env_class=TradingEnv, **env_kwargs):
    """Evaluate agent on a specific date range slice."""
    dates = df['date'].dt.tz_localize(None) if df['date'].dt.tz is not None else df['date']
    s, e  = pd.Timestamp(start), pd.Timestamp(end)
    sub   = df[(dates >= s) & (dates <= e)].copy().reset_index(drop=True)
    if len(sub) < 10:
        return None, None
    env = env_class(sub, initial_balance=CONFIG['INITIAL_BALANCE'],
                    tc=CONFIG['TRANSACTION_COST'], slippage=CONFIG['SLIPPAGE'],
                    **env_kwargs)
    hist, val = agent.evaluate(env)
    return hist, val


print('CRISIS PERIOD STRESS TEST')
print('Evaluating on sub-periods within full dataset')
print()

crisis_summary = {}
for period_name, (start, end) in CONFIG['CRISIS_PERIODS'].items():
    row = {}
    for name, agent, env_cls, ekw in [
        ('PPO',         ppo_agent,     TradingEnv,         {}),
        ('CVaR-PPO',    cvar_agent,    TradingEnv,         {}),
        ('Sortino-PPO', sortino_agent, SortinoTradingEnv,  {'sortino_lambda': CONFIG['SORTINO_LAMBDA']}),
    ]:
        hist, val = eval_on_period(agent, data, start, end, env_cls, **ekw)
        if hist:
            m = calculate_metrics(hist, CONFIG['INITIAL_BALANCE'])
            row[name] = {'return': m['total_return'], 'max_dd': m['max_drawdown'], 'sortino': m['sortino']}

    # Buy & Hold
    dates = data['date'].dt.tz_localize(None) if data['date'].dt.tz is not None else data['date']
    sub_bh = data[(dates >= pd.Timestamp(start)) & (dates <= pd.Timestamp(end))].reset_index(drop=True)
    if len(sub_bh) >= 10:
        bh_h, _ = buy_and_hold(sub_bh, CONFIG['INITIAL_BALANCE'])
        m = calculate_metrics(bh_h, CONFIG['INITIAL_BALANCE'])
        row['Buy & Hold'] = {'return': m['total_return'], 'max_dd': m['max_drawdown'], 'sortino': m['sortino']}

    crisis_summary[period_name] = row
    print(f'{period_name} ({start} to {end}):')
    for mname, vals in row.items():
        print(f'  {mname:15s}: Return={vals["return"]*100:+6.2f}%,  '
              f'MaxDD={vals["max_dd"]*100:+6.2f}%,  Sortino={vals["sortino"]:+.3f}')
    print()

## 18. Multi-Asset Evaluation

In [ ]:
print('MULTI-ASSET EVALUATION')
print('Testing pre-trained models (trained on SPY) on other assets')
print('Note: No re-training. This tests generalization.')
print()

multi_results = {}
for sym in CONFIG['SYMBOLS']:
    sym_data = featured_data[sym]
    _, _, sym_test = split_by_date(sym_data, CONFIG['TRAIN_END'], CONFIG['VAL_END'])

    if len(sym_test) < 10:
        continue

    sym_row = {}
    for name, agent, env_cls, ekw in [
        ('PPO',         ppo_agent,     TradingEnv,        {}),
        ('CVaR-PPO',    cvar_agent,    TradingEnv,        {}),
        ('Sortino-PPO', sortino_agent, SortinoTradingEnv, {'sortino_lambda': CONFIG['SORTINO_LAMBDA']}),
    ]:
        e = env_cls(sym_test, initial_balance=CONFIG['INITIAL_BALANCE'],
                    tc=CONFIG['TRANSACTION_COST'], slippage=CONFIG['SLIPPAGE'], **ekw)
        hist, _ = agent.evaluate(e)
        m = calculate_metrics(hist, CONFIG['INITIAL_BALANCE'])
        sym_row[name] = m

    bh_h, _ = buy_and_hold(sym_test, CONFIG['INITIAL_BALANCE'])
    sym_row['Buy & Hold'] = calculate_metrics(bh_h, CONFIG['INITIAL_BALANCE'])
    multi_results[sym] = sym_row

    print(f'{sym} test results:')
    for mname, m in sym_row.items():
        print(f'  {mname:15s}: Return={m["total_return"]*100:+6.2f}%,  '
              f'Sharpe={m["sharpe"]:+.3f},  MaxDD={m["max_drawdown"]*100:+6.2f}%,  '
              f'CVaR95={m["cvar_95"]*100:+.3f}%')
    print()

## 19. Visualization — Test Set Comparison

In [ ]:
# Portfolio value over time
fig = go.Figure()
plot_data = [
    ('PPO',         ppo_hist_test,     'blue'),
    ('CVaR-PPO',    cvar_hist_test,    'red'),
    ('Sortino-PPO', sortino_hist_test, 'green'),
    ('Buy & Hold',  bh_test_hist,      'orange'),
]
for name, hist, color in plot_data:
    fig.add_trace(go.Scatter(
        y=[h['portfolio_value'] for h in hist],
        name=name, mode='lines', line=dict(color=color, width=2)
    ))

fig.add_hline(y=CONFIG['INITIAL_BALANCE'], line_dash='dash', line_color='gray')
fig.update_layout(title='Portfolio Value — Test Set (2023-2024)',
                  xaxis_title='Trading Day', yaxis_title='Portfolio Value ($)',
                  template='plotly_white', width=1000, height=480)
fig.show()

# Key metrics bar chart
fig2 = make_subplots(rows=2, cols=3,
                     subplot_titles=('Total Return', 'Sharpe Ratio', 'Sortino Ratio',
                                     'Max Drawdown', 'CVaR-95', 'Calmar Ratio'))

method_names = list(metrics.keys())
bar_colors   = ['blue', 'red', 'green', 'orange']
metric_keys  = [('total_return',1,1), ('sharpe',1,2), ('sortino',1,3),
                ('max_drawdown',2,1), ('cvar_95',2,2), ('calmar',2,3)]

for mkey, r, c in metric_keys:
    vals = [metrics[m][mkey] for m in method_names]
    fig2.add_trace(
        go.Bar(x=method_names, y=vals, marker_color=bar_colors, showlegend=False),
        row=r, col=c
    )

fig2.update_layout(title='Performance Metrics Comparison', template='plotly_white',
                   width=1100, height=600)
fig2.show()

## 20. Drawdown Analysis

In [ ]:
fig = go.Figure()
for name, hist, color in plot_data:
    pv   = np.array([h['portfolio_value'] for h in hist])
    peak = np.maximum.accumulate(pv)
    dd   = (pv - peak) / (peak + 1e-10) * 100
    fig.add_trace(go.Scatter(
        y=dd, name=name, mode='lines',
        line=dict(color=color, width=1.5), fill='tozeroy', fillcolor=color.replace(')', ', 0.08)').replace('rgb', 'rgba') if 'rgb' in color else color
    ))

fig.update_layout(
    title='Drawdown (%) — Test Set',
    xaxis_title='Trading Day', yaxis_title='Drawdown (%)',
    template='plotly_white', width=1000, height=380
)
fig.show()

print('Max Drawdown comparison:')
for name in method_names:
    dd = metrics[name]['max_drawdown']
    print(f'  {name:15s}: {dd*100:+.2f}%')

## 21. Return Distribution Analysis

In [ ]:
fig = go.Figure()
for name, hist, color in plot_data:
    pv  = np.array([h['portfolio_value'] for h in hist])
    ret = np.diff(pv) / (pv[:-1] + 1e-10) * 100
    fig.add_trace(go.Histogram(
        x=ret, name=name, opacity=0.6,
        marker_color=color, nbinsx=40
    ))

fig.update_layout(
    title='Daily Return Distribution — Test Set',
    xaxis_title='Daily Return (%)', yaxis_title='Count',
    barmode='overlay', template='plotly_white',
    width=900, height=400
)
fig.show()

print('Return distribution (daily, %):')
print(f'{"Method":15s}  {"Mean":>8s}  {"Std":>8s}  {"VaR-95":>8s}  {"CVaR-95":>8s}')
for name, hist, _ in plot_data:
    pv  = np.array([h['portfolio_value'] for h in hist])
    ret = np.diff(pv) / (pv[:-1] + 1e-10) * 100
    v95  = np.percentile(ret, 5)
    cv95 = ret[ret <= v95].mean() if len(ret[ret <= v95]) > 0 else v95
    print(f'{name:15s}  {ret.mean():+8.4f}  {ret.std():8.4f}  {v95:+8.4f}  {cv95:+8.4f}')

## 22. Save Results and Models

In [ ]:
# Save all models
torch.save(ppo_agent.policy.state_dict(),     'ppo_model.pth')
torch.save(cvar_agent.policy.state_dict(),    'cvar_ppo_model.pth')
torch.save(sortino_agent.policy.state_dict(), 'sortino_ppo_model.pth')

# Save feature normalization params for demo inference
env_ref = train_env_ppo
norm_params = {
    'feat_cols':  env_ref.feat_cols,
    'feat_mean':  env_ref.feat_mean.tolist(),
    'feat_std':   env_ref.feat_std.tolist(),
    'state_dim':  int(env_ref.observation_space.shape[0]),
    'action_dim': int(env_ref.action_space.shape[0]),
}
with open('norm_params.json', 'w') as f:
    json.dump(norm_params, f)

# Save test results
results_out = {
    'config': {k: str(v) for k, v in CONFIG.items()},
    'test_metrics': {k: {mk: float(mv) for mk, mv in v.items()} for k, v in metrics.items()},
    'crisis_summary': {
        p: {m: {k: float(v) for k, v in vals.items()} for m, vals in row.items()}
        for p, row in crisis_summary.items()
    }
}
with open('results.json', 'w') as f:
    json.dump(results_out, f, indent=2)

print('Saved files:')
print('  ppo_model.pth')
print('  cvar_ppo_model.pth')
print('  sortino_ppo_model.pth')
print('  norm_params.json  <- needed for demo inference')
print('  results.json')
print('  test_results.csv')

## 23. Conclusion

In [ ]:
print('FINAL SUMMARY')
print(f'Data: {CONFIG["PRIMARY_SYMBOL"]} | {CONFIG["START_DATE"]} to {CONFIG["END_DATE"]}')
print(f'Test period: {test_data["date"].iloc[0].date()} to {test_data["date"].iloc[-1].date()}')
print(f'Initial balance: ${CONFIG["INITIAL_BALANCE"]:,}')
print()

print('METHOD COMPARISON (Test Set)')
print(f'{"Method":15s}  {"Return":>8s}  {"Sharpe":>8s}  {"Sortino":>8s}  {"Calmar":>8s}  {"MaxDD":>8s}  {"CVaR-95":>8s}')
for name in method_names:
    m = metrics[name]
    print(f'{name:15s}  {m["total_return"]*100:+7.2f}%  '
          f'{m["sharpe"]:8.4f}  {m["sortino"]:8.4f}  '
          f'{m["calmar"]:8.4f}  {m["max_drawdown"]*100:+7.2f}%  '
          f'{m["cvar_95"]*100:+7.4f}%')
print()

best_return  = max(metrics, key=lambda k: metrics[k]['total_return'])
best_sharpe  = max(metrics, key=lambda k: metrics[k]['sharpe'])
best_sortino = max(metrics, key=lambda k: metrics[k]['sortino'])
best_calmar  = max(metrics, key=lambda k: metrics[k]['calmar'])
best_dd      = max(metrics, key=lambda k: metrics[k]['max_drawdown'])  # Least negative
best_cvar    = max(metrics, key=lambda k: metrics[k]['cvar_95'])       # Least negative

print(f'Best Total Return:  {best_return}')
print(f'Best Sharpe Ratio:  {best_sharpe}')
print(f'Best Sortino Ratio: {best_sortino}')
print(f'Best Calmar Ratio:  {best_calmar}')
print(f'Best Max Drawdown:  {best_dd}')
print(f'Best CVaR-95:       {best_cvar}')
print()
print('Observations:')
cvar_dd  = metrics['CVaR-PPO']['max_drawdown']
ppo_dd   = metrics['PPO']['max_drawdown']
sort_dd  = metrics['Sortino-PPO']['max_drawdown']
if abs(cvar_dd) < abs(ppo_dd):
    print('  CVaR-PPO reduces maximum drawdown vs PPO (tail risk constraint works)')
if abs(sort_dd) < abs(ppo_dd):
    print('  Sortino-PPO reduces maximum drawdown vs PPO (downside penalty works)')
if metrics['Buy & Hold']['total_return'] > metrics['PPO']['total_return']:
    print('  Buy & Hold outperforms RL methods in this bull-market test period')
    print('  RL methods show their advantage in risk-adjusted metrics and crisis periods')
print()
print('Future improvements:')
print('  1. Longer training (300+ episodes)')
print('  2. LSTM-based actor for temporal dependencies')
print('  3. Multi-asset portfolio (joint action space)')
print('  4. Sentiment / macro features (VIX, yield curve)')